# LegalDataPlatform - Analytical queries

This notebook exercises the data the pipelines produce. Run it **after** you've executed at least one round of each flow:

```bash
make pipeline       # legal_documents from CSV
make sec-edgar      # real filings from SEC EDGAR
make gleif          # real Legal Entity Identifiers from GLEIF
make commercial     # contracts + transactions from CSV
```

What you'll see:

1. Cross-source counts across the Medallion layers.
2. Point-in-time queries against the SCD2 `dim_counterparty` dimension.
3. Partition-pruning demonstration via EXPLAIN.
4. Data-quality metrics from the quarantine layer.
5. Materialized-view scans that should run in sub-millisecond range.

In [ ]:
%load_ext sql
import os

from dotenv import load_dotenv
load_dotenv('../.env')

CONN_STR = (
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)
%sql $CONN_STR

import pandas as pd
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 50)

## 1. Volume per source

After running all flows, `legal_documents` should contain rows from both `legal_csv` (seed) and `sec_edgar` (real API).

In [ ]:
%%sql
SELECT source_system,
       document_type,
       count(*) AS documents,
       min(document_date) AS earliest,
       max(document_date) AS latest
  FROM legal_documents
 GROUP BY source_system, document_type
 ORDER BY source_system, documents DESC

## 2. Counterparties sourced from GLEIF vs CSV seed

Counterparties can come from two origins:
- The CSV seed (synthetic companies with `CP-<uuid>` external ids).
- GLEIF (real companies with `LEI-<20char>` external ids).

Both pass through the same validation + SCD2 pipeline.

In [ ]:
%%sql
SELECT
    CASE
        WHEN external_id LIKE 'LEI-%' THEN 'GLEIF'
        WHEN external_id LIKE 'CP-%'  THEN 'seed'
        ELSE 'other'
    END AS origin,
    country_code,
    count(*) AS counterparties,
    count(*) FILTER (WHERE tax_id IS NOT NULL) AS with_tax_id
  FROM counterparties
 GROUP BY 1, country_code
 ORDER BY counterparties DESC
 LIMIT 15

## 3. SCD2 point-in-time query

`dim_counterparty` preserves history. Ask the same question at different dates to see time-travel in action.

In [ ]:
%%sql
WITH sample AS (
    SELECT external_id FROM dim_counterparty LIMIT 1
)
SELECT
    dc.external_id,
    dc.name,
    dc.risk_score,
    dc.valid_from,
    dc.valid_to,
    dc.is_current
FROM dim_counterparty dc
JOIN sample s ON s.external_id = dc.external_id
ORDER BY dc.valid_from

## 4. Partition pruning demonstration

Queries with a date range filter should only scan the matching partitions. Check the EXPLAIN plan.

In [ ]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT count(*)
  FROM transactions
 WHERE transaction_date >= CURRENT_DATE - INTERVAL '30 days'

## 5. Top-k query with covering index

This query is what the revenue-by-counterparty API would call. The `ix_txn_counterparty_recent` index is a covering index with `INCLUDE (amount, currency)`, so the planner can answer this as an index-only scan.

In [ ]:
%%sql
SELECT c.name, c.country_code,
       count(t.id) AS txn_count,
       round(sum(t.amount)::numeric, 2) AS revenue
  FROM transactions t
  JOIN counterparties c ON c.id = t.counterparty_id
 WHERE t.transaction_date >= CURRENT_DATE - INTERVAL '90 days'
   AND t.transaction_type = 'INVOICE'
 GROUP BY c.id, c.name, c.country_code
 ORDER BY revenue DESC
 LIMIT 10

## 6. Materialized-view scan

`mv_monthly_revenue_per_counterparty` is the "Gold" layer — pre-aggregated monthly revenue. Scans are sub-millisecond because the aggregation is already done.

In [ ]:
%%sql
SELECT month,
       count(DISTINCT counterparty_external_id) AS counterparties,
       sum(transaction_count) AS total_txns,
       round(sum(total_amount)::numeric, 2) AS total_revenue
  FROM mv_monthly_revenue_per_counterparty
 WHERE currency = 'USD'
 GROUP BY month
 ORDER BY month DESC
 LIMIT 12

## 7. Slow-query tracking with `pg_stat_statements`

Which queries has the database itself flagged as expensive? This is the first place to look when tuning.

In [ ]:
%%sql
SELECT
    calls,
    round(mean_exec_time::numeric, 2) AS mean_ms,
    round(total_exec_time::numeric, 2) AS total_ms,
    round(100.0 * shared_blks_hit / nullif(shared_blks_hit + shared_blks_read, 0), 2) AS cache_hit_pct,
    left(query, 80) AS query
  FROM pg_stat_statements
 WHERE calls > 5
   AND query NOT LIKE '%pg_stat_statements%'
 ORDER BY mean_exec_time DESC
 LIMIT 10

## 8. Partition inventory and sizes

How big is each monthly partition? When one starts to outgrow the others, it's a signal that retention policy needs attention.

In [ ]:
%%sql
SELECT
    c.relname AS partition,
    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,
    pg_size_pretty(pg_relation_size(c.oid)) AS table_size,
    pg_size_pretty(pg_indexes_size(c.oid)) AS indexes_size,
    s.n_live_tup AS rows
FROM pg_inherits i
JOIN pg_class c ON c.oid = i.inhrelid
JOIN pg_stat_user_tables s ON s.relid = c.oid
WHERE i.inhparent IN ('transactions'::regclass, 'legal_documents'::regclass)
ORDER BY pg_total_relation_size(c.oid) DESC
LIMIT 20

## What this demonstrates

- The pipelines populate the schema with data from multiple real sources.
- SCD2 time-travel works on real records.
- Partition pruning is active (visible in the EXPLAIN plan).
- Indexes are being used (cache hit ratio > 99% on warm queries).
- Materialized views deliver the sub-millisecond latency you'd expect.
- `pg_stat_statements` gives you the baseline for tuning decisions.

Compare these against [docs/evidence/benchmarks.md](../docs/evidence/benchmarks.md) to see how the numbers hold on your specific machine.